# Downscale typhoon rain with AI

**ERA5** is a global record of past weather. It stores rain in squares about 25 km wide, once per hour.
That is too coarse to see where rain fell inside a province.

**spateGAN-ERA5** is an AI model that turns those 25 km squares into **2 km squares every 10 minutes**.
In this notebook you run it yourself on one of three typhoons that hit the Philippines:

| Typhoon | When | Where the box is |
|---|---|---|
| Nando (Ragasa) | 20 to 24 September 2025 | Ilocos coast |
| Kristine (Trami) | 21 to 25 October 2024 | Bicol |
| Tino (Kalmaegi) | 2 to 6 November 2025 | Cebu |

**How to use this notebook**
1. Optional: for a faster run, choose *Runtime > Change runtime type > T4 GPU*. The free CPU also works.
2. Run the steps from top to bottom: click the run button on each step, or press Shift+Enter.
3. Wait until a step is done before you start the next one.

In [ ]:
# @title Step 1: Install spateGAN (about 1 minute)
import os, sys
from pathlib import Path

WORK = Path("/content") if Path("/content").exists() else Path.cwd()
REPO = WORK / "spateGAN_ERA5"
if not REPO.exists():
    !git clone -q https://github.com/LGlawion/spateGAN_ERA5.git {REPO}
    !git -C {REPO} checkout -q 35e4d869f349ee5f4b1a95986828ba48d935898c
%cd {REPO}
%pip install -q einops netCDF4 pyproj cartopy
print("Done.")

## Step 2: Choose a typhoon

spateGAN makes a 2 km map for a square about **290 km wide**. Each typhoon has its square where its rain hit hardest.

*Seed*: the model draws one possible version of the fine detail. A different seed gives another version that is just as likely.

In [ ]:
# @title Step 2: Choose a typhoon { run: "auto" }
TYPHOON = "Nando, September 2025"  # @param ["Nando, September 2025", "Kristine, October 2024", "Tino, November 2025"]
SEED = 10  # @param {type:"integer"}

TYPHOONS = {  # data file, box centre (latitude, longitude), name and place for titles
    "Nando, September 2025": ("nando_era5.nc", (17.8, 120.9), "Typhoon Nando", "Ilocos coast"),
    "Kristine, October 2024": ("kristine_era5.nc", (13.6, 123.4), "Severe Tropical Storm Kristine", "Bicol"),
    "Tino, November 2025": ("tino_era5.nc", (10.4, 124.0), "Typhoon Tino", "Cebu"),
}
FILE, (CENTER_LAT, CENTER_LON), NAME, PLACE = TYPHOONS[TYPHOON]

import urllib.request
DATA = WORK / FILE
if not DATA.exists():
    urllib.request.urlretrieve("https://github.com/ruzcko/precipitation-downscaling/raw/main/data/" + FILE, DATA)

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

HALF = 1.3  # about 145 km in degrees
fig = plt.figure(figsize=(4, 5.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([116, 128, 4.5, 21.5])
ax.add_feature(cfeature.LAND.with_scale("50m"), color="#e6e6e6")
ax.coastlines("50m", linewidth=0.6, color="#555555")
ax.plot([CENTER_LON - HALF, CENTER_LON + HALF, CENTER_LON + HALF, CENTER_LON - HALF, CENTER_LON - HALF],
        [CENTER_LAT - HALF, CENTER_LAT - HALF, CENTER_LAT + HALF, CENTER_LAT + HALF, CENTER_LAT - HALF],
        color="#1f4e8c", linewidth=2)
ax.set_title(f"{NAME}: {PLACE}")
plt.show()

In [ ]:
# @title Step 3: Run the AI downscaling (1 to 5 minutes)
import glob
import torch
import yaml
from src.spategan_era5.pipeline import run_downscaling_pipeline

OUTPUT = WORK / "output" / DATA.stem
config = yaml.safe_load(open("config/config.yml"))
config["data"].update(input_path=str(DATA), output_latlon_path=str(OUTPUT), save_model_input=False)
config["domain"].update(center_lat=CENTER_LAT, center_lon=CENTER_LON)
config["processing"].update(device="cuda" if torch.cuda.is_available() else "cpu", seed=SEED)
config["plotting"]["precipitation_sums"] = False

print("Running on", config["processing"]["device"].upper())
run_downscaling_pipeline(config, REPO)
RESULT = max(glob.glob(str(OUTPUT / "spateGAN_ERA5_latlon_*.nc")), key=os.path.getmtime)
print("Saved:", RESULT)

In [ ]:
# @title Step 4: Compare coarse and fine with a slider
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.patheffects as pe
from matplotlib.colors import BoundaryNorm, ListedColormap
from ipywidgets import IntSlider, interact

fine = xr.open_dataset(RESULT).precipitation              # mm per hour, every 10 min
fine = fine.isel(time=fine.notnull().any(("lat", "lon")).values)
era5 = xr.open_dataset(DATA)
coarse = (era5.cp + era5.lsp) * 1000                       # metres per hour -> mm per hour
LON0, LON1 = float(fine.lon.min()), float(fine.lon.max())
LAT0, LAT1 = float(fine.lat.min()), float(fine.lat.max())
coarse = coarse.sel(lat=slice(LAT1 + 0.25, LAT0 - 0.25), lon=slice(LON0 - 0.25, LON1 + 0.25))

LEVELS = [0.5, 1, 2, 5, 10, 20, 50, 100]
CMAP = ListedColormap(plt.cm.Blues(np.linspace(0.35, 1, len(LEVELS) - 1)))
CMAP.set_over(plt.cm.Blues(1.0))
CMAP.set_under("none")
NORM = BoundaryNorm(LEVELS, CMAP.N)


TOWNS = {"Laoag": (18.20, 120.59), "Vigan": (17.57, 120.39), "Bangued": (17.60, 120.62),
         "Aparri": (18.36, 121.64), "Tuguegarao": (17.61, 121.73), "Ilagan": (17.15, 121.89),
         "San Fernando": (16.62, 120.32),
         "Naga": (13.62, 123.19), "Legazpi": (13.14, 123.74), "Iriga": (13.42, 123.41),
         "Daet": (14.11, 122.95), "Virac": (13.58, 124.23), "Sorsogon": (12.97, 124.00),
         "Cebu City": (10.31, 123.89), "Toledo": (10.38, 123.64), "Bogo": (11.05, 124.00),
         "Tagbilaran": (9.65, 123.85), "Ormoc": (11.01, 124.61), "Maasin": (10.13, 124.84),
         "Dumaguete": (9.31, 123.31)}


def ph_time(t):
    return (pd.Timestamp(t) + pd.Timedelta(hours=8)).strftime("%d %b %Y, %H:%M PHT")


def make_figure():
    fig, axes = plt.subplots(1, 2, figsize=(10, 5.6), subplot_kw={"projection": ccrs.PlateCarree()})
    fig.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.17, wspace=0.05)
    meshes = []
    for ax, da, title in [(axes[0], coarse, "ERA5: 25 km, every hour"),
                          (axes[1], fine, "spateGAN: 2 km, every 10 minutes")]:
        ax.set_extent([LON0, LON1, LAT0, LAT1])
        ax.add_feature(cfeature.LAND.with_scale("10m"), color="#e6e6e6")
        meshes.append(ax.pcolormesh(da.lon, da.lat, da.isel(time=0), cmap=CMAP, norm=NORM,
                                    shading="nearest", transform=ccrs.PlateCarree()))
        ax.coastlines("10m", linewidth=0.6, color="#555555")
        for name, (la, lo) in TOWNS.items():
            if LAT0 < la < LAT1 and LON0 < lo < LON1:
                ax.plot(lo, la, "o", color="#222222", markersize=3)
                ax.text(lo + 0.03, la + 0.03, name, fontsize=8, color="#222222",
                        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
        ax.set_title(title)
    cax = fig.add_axes([0.25, 0.08, 0.5, 0.03])
    fig.colorbar(meshes[1], cax=cax, orientation="horizontal", label="Rain (mm per hour)", extend="max")
    suptitle = fig.suptitle("")

    def update(i):
        t = fine.time.values[i]
        hour = pd.Timestamp(t).floor("h")
        meshes[0].set_array(coarse.sel(time=hour, method="nearest").values)
        meshes[1].set_array(fine.isel(time=i).values)
        suptitle.set_text(f"{NAME}, {PLACE}: {ph_time(t)}")

    return fig, update


def show(frame):
    fig, update = make_figure()
    update(frame)
    plt.show()


interact(show, frame=IntSlider(value=0, min=0, max=fine.sizes["time"] - 1, step=1,
                               continuous_update=False, description="Time step"));

In [ ]:
# @title Step 5: Save a GIF of the wettest 24 hours
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

peak = int(np.argmax(fine.mean(("lat", "lon")).values))
start = max(0, peak - 72)
frames = range(start, min(fine.sizes["time"], start + 144))   # 144 x 10 min = 24 hours

fig, update = make_figure()
GIF = WORK / f"{DATA.stem.replace('_era5', '')}_downscaled.gif"
FuncAnimation(fig, update, frames=frames).save(GIF, writer=PillowWriter(fps=8), dpi=72)
plt.close(fig)
display(Image(filename=str(GIF)))
print("Saved:", GIF, "(open the Files panel on the left to download it)")

## Try it yourself

- Go back to **Step 2**, pick another typhoon, and run Steps 3 to 5 again.
- Change the **seed** and run again. The big picture stays the same, but the small storm cells move. The model draws one possible version of the detail, not the one true answer.

*Model: Glawion et al. (2025), "Global spatio-temporal ERA5 precipitation downscaling to km and sub-hourly scale using generative AI", npj Climate and Atmospheric Science. Code: github.com/LGlawion/spateGAN_ERA5 (MIT). Data: ERA5, Copernicus Climate Change Service, via Google ARCO-ERA5.*